# 05 — Assemble training panel

Join the existing PIT panel (CRSP + Sharadar) with macro indicators, aux text
features (`text_novelty`, `days_since_filing`, `doc_count_7d`), and forward-return
labels. Output: `data/processed/training_panel/year=YYYY/part-0.parquet`,
daily cadence, partitioned by year. Consumed by notebook 06 (ranker,
Friday-filtered) and notebook 07+ (RL agent, daily).

**Spec:** `docs/superpowers/specs/2026-05-16-feature-assembly-design.md`.

**Resumable:** existing year-shards are skipped on re-run. Delete a year's
`part-0.parquet` to re-do.

PCA text features are *not* in this panel — they're derived at ranker time
from `finbert_stockday_embed` + the walk's `pca.joblib`. See spec §6.

## A. Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from src.utils.io import processed_dir, repo_root
from src.utils.features import (
    pivot_macro_wide,
    compute_forward_returns,
    compute_text_novelty,
    compute_days_since_filing,
    compute_doc_count_window,
)

PANEL_DIR = processed_dir() / 'panel'
MACRO_PATH = processed_dir() / 'macro.parquet'
EMBED_DIR = processed_dir() / 'finbert_stockday_embed'
EDGAR_INDEX_PATH = processed_dir() / 'edgar_index.parquet'
UNIVERSE_PATH = processed_dir() / 'universe_ids.parquet'
OUT_DIR = processed_dir() / 'training_panel'
SUMMARY_PATH = repo_root() / 'reports' / 'metrics' / 'feature_assembly_summary.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

# Validation thresholds (spec §8).
MAX_TEXT_NOVELTY_NAN_RATE = 0.05
MAX_DAYS_SINCE_FILING_NAN_RATE = 0.10

YEARS = sorted({int(p.name.split('=')[1]) for p in PANEL_DIR.glob('year=*')})
print(f'panel years: {YEARS[0]}..{YEARS[-1]} ({len(YEARS)} years)')
print(f'out_dir: {OUT_DIR}')

## B. Load shared inputs (universe, macro, filings)

Loaded once outside the per-year loop. Universe + filings are small; the
embed shards are accessed per-year inside the loop to keep peak memory bounded.

In [ ]:
universe_ids = pd.read_parquet(UNIVERSE_PATH)
universe_ids['permno'] = universe_ids['permno'].astype('Int64')
universe_ids['date_out'] = universe_ids['date_out'].fillna(pd.Timestamp('2099-12-31'))
print(f'universe_ids: {len(universe_ids)} rows, {universe_ids["permno"].nunique()} permnos')

macro_long = pd.read_parquet(MACRO_PATH)
print(f'macro: {len(macro_long)} rows, series={sorted(macro_long["series"].unique())}')

edgar_index = pd.read_parquet(EDGAR_INDEX_PATH)
edgar_index['filing_date'] = pd.to_datetime(edgar_index['filing_date'])
# Restrict to ciks in our universe — drops noise from the broader EDGAR pull.
edgar_index = edgar_index[edgar_index['cik'].isin(universe_ids['cik'])]
print(f'edgar_index (universe-filtered): {len(edgar_index)} filings')

## C. Per-year assembly helper

`assemble_year(year)` does: read `panel/year=YYYY/`, universe-filter, left-join
macro (pivoted + ffilled over that year's date range), left-join cik via
`universe_ids` (so we can attach filings later), then attach aux text features
(`text_novelty` from the year's embed shard; `days_since_filing` + `doc_count_7d`
from `edgar_index`). Returns the per-year dataframe ready for labels (cell D)
and persist (cell F).

In [ ]:
def _read_year_embed(year: int) -> pd.DataFrame:
    """Read finbert_stockday_embed shards for one year. Also peeks at year-1's
    last 7 days so the year's earliest rows can find a t-7 predecessor."""
    shards = sorted(EMBED_DIR.glob(f'year={year}/*.parquet'))
    prior_shards = sorted(EMBED_DIR.glob(f'year={year - 1}/*.parquet'))
    frames = []
    for s in shards:
        frames.append(pd.read_parquet(s, columns=['permno', 'date', 'vec']))
    if prior_shards:
        for s in prior_shards:
            df = pd.read_parquet(s, columns=['permno', 'date', 'vec'])
            df = df[df['date'] >= pd.Timestamp(f'{year - 1}-12-24')]  # last week of prior year
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=['permno', 'date', 'vec'])
    embed = pd.concat(frames, ignore_index=True)
    embed['date'] = pd.to_datetime(embed['date'])
    return embed


def assemble_year(year: int) -> pd.DataFrame:
    # 1. Read panel year.
    panel_shards = sorted(PANEL_DIR.glob(f'year={year}/*.parquet'))
    base = pd.concat([pd.read_parquet(s) for s in panel_shards], ignore_index=True)
    base['date'] = pd.to_datetime(base['date'])

    # 2. Universe filter (interval merge, same logic as src/utils/pca.py).
    intervals = universe_ids[['permno', 'cik', 'date_in', 'date_out']].copy()
    intervals['permno'] = intervals['permno'].astype('int64')
    merged = base.merge(intervals, on='permno', how='inner')
    in_window = (merged['date'] >= merged['date_in']) & (merged['date'] <= merged['date_out'])
    base = (merged[in_window]
            .drop(columns=['date_in', 'date_out'])
            .drop_duplicates(subset=['permno', 'date']))

    # 3. Macro: pivot, ffill to this year's calendar dates, left-join.
    year_dates = pd.DatetimeIndex(sorted(base['date'].unique()))
    macro_w = pivot_macro_wide(macro_long, ffill_dates=year_dates)
    base = base.merge(macro_w, on='date', how='left')

    # 4. Aux text features.
    embed = _read_year_embed(year)
    novelty = compute_text_novelty(embed, lookback_days=7)
    # Drop the borrowed t-1y rows from the output (we only kept them for lookups).
    novelty = novelty[novelty['date'].dt.year == year]
    base = base.merge(novelty, on=['permno', 'date'], how='left')

    dsf = compute_days_since_filing(edgar_index, base[['permno', 'cik', 'date']])
    base = base.merge(dsf, on=['permno', 'date'], how='left')

    docc = compute_doc_count_window(edgar_index, base[['permno', 'cik', 'date']], window_days=7)
    base = base.merge(docc, on=['permno', 'date'], how='left')

    return base